# 📊 CONSTRUÇÃO DOS GRÁFICOS EM FORMATO `.html` (v1.0)
### **Estudo de Caso: Análise e Previsão da Produção Mensal de Petróleo no Brasil**

Este notebook documenta o pipeline de geração de gráficos interativos em formato HTML utilizando a biblioteca **Plotly**. O design visual foi projetado sob medida para se integrar de forma fluida à identidade corporativa escura da **DOCHMO Analytics**.

**Etapas do Notebook:**
1. Definição do tema padrão compartilhado (`apply_dochmo_theme`).
2. Carregamento dos dados brutos ANP (Janeiro/2001 a Maio/2025).
3. Gráfico 01: Série Temporal Original.
4. Gráfico 02: Decomposição Sazonal Multiplicativa (Subplots).
5. Gráfico 03: Comparação de Modelos Preditivos no Horizonte de Teste (*Out-of-Sample*).

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Desativa alertas de convergência numérica para manter a saída limpa
warnings.filterwarnings("ignore")

# ==============================================================================
# CONFIGURAÇÕES DO TEMA CORPORATIVO DOCHMO
# ==============================================================================
BG_TRANSPARENT = "rgba(0,0,0,0)"
PLOT_BG_SOLID = "#0C1426"       # Fundo navy-blue escuro para os plots
COLOR_GOLD = "#D7B56D"           # Dourado institucional para a série principal
COLOR_GOLD_ALPHA = "rgba(215, 181, 109, 0.5)"  # Dourado translúcido para marcadores
COLOR_GOLD_IC = "rgba(215, 181, 109, 0.12)"    # Sombra do Intervalo de Confiança
COLOR_WHITE = "#F8F9FA"          # Branco puro para os Dados Reais (Teste)
COLOR_WHITE_SOFT = "rgba(248, 249, 250, 0.8)"  # Branco suave para tendência/sazonalidade
COLOR_WHITE_FAINT = "rgba(248, 249, 250, 0.4)" # Branco/cinza fraco para treino
COLOR_WHITE_AXIS = "rgba(248, 249, 250, 0.65)" # Rótulos dos eixos e ranhuras
COLOR_BORDER = "rgba(248, 249, 250, 0.15)"     # Linha base dos eixos
COLOR_GREEN_V1 = "#4CAF50"       # Verde para a curva Holt-Winters
GRID_COLOR = "rgba(215, 181, 109, 0.1)"         # Grid dourada sutil

FONT_TITLE = "Playfair Display"  # Serif sofisticado para cabeçalhos
FONT_AXIS = "Inter, sans-serif"  # Sans-serif legível para eixos e tooltips

def apply_dochmo_theme(fig, height=480):
    """
    Aplica as especificações de estilo corporativo da DOCHMO a uma figura Plotly.
    Mantém o fundo do container transparente e define o plot interno como navy escuro.
    """
    fig.update_layout(
        paper_bgcolor=BG_TRANSPARENT,
        plot_bgcolor=PLOT_BG_SOLID,
        font=dict(family=FONT_AXIS, color=COLOR_WHITE_AXIS, size=13),
        height=height,
        margin=dict(l=65, r=30, t=55, b=55),
        title=dict(
            font=dict(family=FONT_TITLE, color=COLOR_GOLD, size=20),
            x=0.5
        ),
        legend=dict(
            font=dict(color=COLOR_WHITE, size=12),
            bgcolor=BG_TRANSPARENT
        ),
        hoverlabel=dict(
            bgcolor="#0F1C36",
            font=dict(family=FONT_AXIS, color=COLOR_WHITE, size=13),
            bordercolor=COLOR_GOLD,
        ),
    )
    # Formatação das linhas e grids dos eixos
    fig.update_xaxes(
        showgrid=True, gridcolor=GRID_COLOR,
        showline=True, linecolor=COLOR_BORDER,
        tickfont=dict(color=COLOR_WHITE_AXIS),
        zeroline=False,
    )
    fig.update_yaxes(
        showgrid=True, gridcolor=GRID_COLOR,
        showline=False,
        tickfont=dict(color=COLOR_WHITE_AXIS),
        zeroline=False,
    )
    return fig

def write_case_html(fig, path):
    """
    Salva a figura final como um arquivo HTML estático leve, importando a biblioteca
    Plotly.js via CDN para otimizar o tempo de carregamento da página final.
    """
    fig.write_html(
        path,
        include_plotlyjs="cdn",
        full_html=True,
        config={"displayModeBar": False, "responsive": True},
    )
    print(f"[OK] Salvo: {path}")

## 📈 Carregamento e Preparação dos Dados

Lê os dados da planilha Excel do histórico de produção. A série é truncada entre **Janeiro/2001 e Maio/2025**.

In [ ]:
# Caminho absoluto da base de dados excel oficial da ANP
caminho_excel = Path("../dataset/Produção Mensal de Petróleo no Brasil.xlsx")
dados = pd.read_excel(caminho_excel)
dados.columns = ["data", "producao"]

# Parser de data e corte amostral estrito
dados["data"] = pd.to_datetime(dados["data"])
dados = dados[(dados["data"] >= "2001-01-01") & (dados["data"] <= "2025-05-01")]

# Indexação temporal e fixação de frequência para início do mês (MS)
dados.set_index("data", inplace=True)
serie_petroleo = dados["producao"].asfreq("MS")

# DataFrame auxiliar para estrutura de plotagem no Plotly Express
serie_df = serie_petroleo.reset_index()
serie_df.columns = ["data", "valor"]

## 01. Série Temporal Histórica Original

Plota o comportamento macro da produção de petróleo, expondo a forte tendência ascendente impulsionada pelas descobertas do pré-sal e a expressiva sazonalidade mensal.

In [ ]:
# 1. Mapeamento de meses para tooltips amigáveis em português
meses_pt = {1: 'Jan', 2: 'Fev', 3: 'Mar', 4: 'Abr', 5: 'Mai', 6: 'Jun', 
            7: 'Jul', 8: 'Ago', 9: 'Set', 10: 'Out', 11: 'Nov', 12: 'Dez'}
serie_df['data_hover'] = serie_df['data'].dt.month.map(meses_pt) + " de " + serie_df['data'].dt.year.astype(str)

# 2. Criação do gráfico de linha principal
fig_serie = px.line(
    serie_df, x="data", y="valor", 
    title="Série Temporal da Produção Mensal de Petróleo",
    custom_data=['data_hover']
)

# 3. Ajuste fino de trace e hovertemplate
fig_serie.update_traces(
    line_color=COLOR_GOLD, line_width=2.5,
    hovertemplate="<b>%{customdata[0]}</b><br>Volume: %{y:,.0f} m³<extra></extra>"
)

# 4. Definição do separador numérico BR (ponto para milhar, vírgula para decimal)
fig_serie.update_layout(separators=",.")

# 5. Aplicação do tema corporativo e exportação
apply_dochmo_theme(fig_serie, height=500)
write_case_html(fig_serie, "../graphics/g01_grafico_serie_original.html")

## 02. Decomposição Sazonal Multiplicativa

Modela a série histórica no domínio do tempo dividindo-a em três componentes estruturais distintos pelo método clássico multiplicativo:
*   **Tendência:** Trajetória de longo prazo.
*   **Sazonalidade:** Flutuações periódicas repetitivas dentro de um ciclo de 12 meses.
*   **Resíduos:** Choques estocásticos imprevisíveis (ruído).

In [ ]:
# Executa a decomposição multiplicativa clássica pelo Statsmodels
decomposicao = seasonal_decompose(serie_petroleo, model="multiplicative")

# Geração de datas estruturadas em formato literal português para tooltips
meses_full_pt = {1: 'Janeiro', 2: 'Fevereiro', 3: 'Março', 4: 'Abril', 5: 'Maio', 6: 'Junho', 
                 7: 'Julho', 8: 'Agosto', 9: 'Setembro', 10: 'Outubro', 11: 'Novembro', 12: 'Dezembro'}
hover_datas = [f"{meses_full_pt[d.month]} de {d.year}" for d in decomposicao.observed.index]

# Inicializa o painel de subplots alinhados verticalmente com eixo X compartilhado
fig_decomp = make_subplots(
    rows=4, cols=1, shared_xaxes=True, 
    subplot_titles=["Observado", "Tendência", "Sazonalidade", "Resíduo"],
    vertical_spacing=0.08
)

# Adiciona o trace Observado (Série Bruta em Dourado)
fig_decomp.add_trace(go.Scatter(
    x=decomposicao.observed.index, y=decomposicao.observed, name="Observado", line=dict(color=COLOR_GOLD),
    customdata=hover_datas,
    hovertemplate="%{customdata}<br><b>Volume: %{y:,.0f} m³</b><extra></extra>"
), row=1, col=1)

# Adiciona o trace de Tendência (Curva suavizada em Branco Suave)
fig_decomp.add_trace(go.Scatter(
    x=decomposicao.trend.index, y=decomposicao.trend, name="Tendência", line=dict(color=COLOR_WHITE_SOFT),
    customdata=hover_datas,
    hovertemplate="%{customdata}<br><b>Tendência: %{y:,.0f} m³</b><extra></extra>"
), row=2, col=1)

# Adiciona o trace Sazonal (Oscilações periódicas em Branco Suave)
fig_decomp.add_trace(go.Scatter(
    x=decomposicao.seasonal.index, y=decomposicao.seasonal, name="Sazonalidade", line=dict(color=COLOR_WHITE_SOFT),
    customdata=hover_datas,
    hovertemplate="%{customdata}<br><b>Índice Sazonal: %{y:.4f}</b><extra></extra>"
), row=3, col=1)

# Adiciona o trace de Resíduos (Dispersão de pontos dourados semitransparentes)
fig_decomp.add_trace(go.Scatter(
    x=decomposicao.resid.index, y=decomposicao.resid, name="Resíduo", mode="markers", marker=dict(color=COLOR_GOLD_ALPHA),
    customdata=hover_datas,
    hovertemplate="%{customdata}<br><b>Resíduo: %{y:.4f}</b><extra></extra>"
), row=4, col=1)

# Configuração individual de formato nos eixos Y de cada componente
fig_decomp.update_yaxes(tickformat=",.0f", row=1, col=1)
fig_decomp.update_yaxes(tickformat=",.0f", row=2, col=1)
fig_decomp.update_yaxes(tickformat=",.3f", row=3, col=1)
fig_decomp.update_yaxes(tickformat=",.3f", row=4, col=1)

fig_decomp.update_layout(
    separators=",.",
    height=780,
    title=dict(text="Decomposição Multiplicativa da Série", x=0.5, font=dict(size=20)),
    showlegend=False,
    hovermode="x unified"
)
apply_dochmo_theme(fig_decomp, height=780)

# Garante que os títulos internos dos subplots herdem a fonte Inter padrão
for annotation in fig_decomp['layout']['annotations']:
    annotation['font'] = dict(family=FONT_AXIS, size=14, color=COLOR_WHITE)

write_case_html(fig_decomp, "../graphics/g02_grafico_decomposicao.html")

## 03. Comparação de Modelos Preditivos no Holdout (*Out-of-Sample*)

Modela a série temporal no ambiente de transformação logarítmica para estabilização de variância (homocedasticidade).
O período de treinamento é definido até Abril de 2021, utilizando os 49 meses restantes para competição preditiva entre o modelo de intervenção estatística clássico **SARIMA(2,1,3)(0,1,1)₁₂** e o benchmark exponencial de **Holt-Winters Aditivo**.

In [ ]:
# Divisão amostral: Treinamento (Jan/2001 - Abr/2021) e Teste/Holdout (Mai/2021 - Mai/2025)
treino = serie_petroleo.loc[:"2021-04-01"]
teste = serie_petroleo.loc["2021-05-01":]

# Estabilização matemática via logaritmo natural
treino_log = np.log(treino)
teste_log = np.log(teste)

# ------------------------------------------------------------------------------
# 1. MODELO CAMPEÃO: SARIMA(2,1,3)(0,1,1)12
# ------------------------------------------------------------------------------
modelo_sarima = SARIMAX(
    treino_log,
    order=(2, 1, 3),
    seasonal_order=(0, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False
)
resultado_sarima = modelo_sarima.fit(disp=False, maxiter=200)

# Extração de projeções pontuais e intervalos de confiança (IC) a 95% estatísticos
previsao_sarima = resultado_sarima.get_forecast(steps=len(teste_log))
sarima_mean = previsao_sarima.predicted_mean
sarima_ci = previsao_sarima.conf_int(alpha=0.05)

# ------------------------------------------------------------------------------
# 2. MODELO DE BENCHMARK: SUAVIZAÇÃO EXPONENCIAL (HOLT-WINTERS)
# ------------------------------------------------------------------------------
modelo_hw = ExponentialSmoothing(
    treino_log, 
    trend='add', 
    seasonal='add', 
    seasonal_periods=12
).fit()
previsao_hw = modelo_hw.forecast(len(teste_log))

# ------------------------------------------------------------------------------
# 3. MONTAGEM DO GRÁFICO COMPARATIVO ENTERPRISE
# ------------------------------------------------------------------------------
hover_treino = [f"{meses_full_pt[d.month]} de {d.year}" for d in treino_log.index]
hover_teste = [f"{meses_full_pt[d.month]} de {d.year}" for d in teste_log.index]

fig_final = go.Figure()

# Adiciona curva de Treino (Branco Suave com Opacidade)
fig_final.add_trace(go.Scatter(
    x=treino_log.index, y=treino_log.values, mode='lines', name='Treino', 
    line=dict(color=COLOR_WHITE_FAINT, width=2),
    customdata=hover_treino,
    hovertemplate="<b>%{customdata}</b><br>Treino: %{y:,.4f}<extra></extra>"
))

# Adiciona dados observados reais (Branco Puro Sólido)
fig_final.add_trace(go.Scatter(
    x=teste_log.index, y=teste_log.values, mode='lines', name='Dados Reais', 
    line=dict(color=COLOR_WHITE, width=2.5),
    customdata=hover_teste,
    hovertemplate="<b>%{customdata}</b><br>Real: %{y:,.4f}<extra></extra>"
))

# Adiciona projeção do SARIMA (Dourado Sólido)
fig_final.add_trace(go.Scatter(
    x=sarima_mean.index, y=sarima_mean.values, mode='lines', name='Previsão SARIMA', 
    line=dict(color=COLOR_GOLD, width=2.5),
    customdata=hover_teste,
    hovertemplate="<b>%{customdata}</b><br>SARIMA: %{y:,.4f}<extra></extra>"
))

# Adiciona área sombreada do Intervalo de Confiança (IC 95%) do SARIMA
fig_final.add_trace(go.Scatter(
    x=list(sarima_ci.index) + list(sarima_ci.index)[::-1],
    y=list(sarima_ci.iloc[:, 1]) + list(sarima_ci.iloc[:, 0])[::-1],
    fill='toself', fillcolor=COLOR_GOLD_IC, line=dict(color='rgba(255,255,255,0)'),
    name='IC 95% SARIMA', showlegend=False,
    hoverinfo='skip'
))

# Adiciona projeção de Holt-Winters (Linha verde tracejada)
fig_final.add_trace(go.Scatter(
    x=previsao_hw.index, y=previsao_hw.values, mode='lines', name='Previsão Holt-Winters', 
    line=dict(color=COLOR_GREEN_V1, dash='dash', width=2.5),
    customdata=hover_teste,
    hovertemplate="<b>%{customdata}</b><br>Holt-Winters: %{y:,.4f}<extra></extra>"
))

# Layout com altura aumentada e posicionamento da legenda com espaçamento estrito
fig_final.update_layout(
    separators=",.",
    height=580,
    title=dict(text="Comparativo Final de Projeções: Real vs SARIMA vs Holt-Winters", font=dict(size=20)),
    yaxis=dict(tickformat=",.4f"),
    legend=dict(orientation="h", yanchor="bottom", y=1.10, xanchor="right", x=1, font=dict(color=COLOR_WHITE)), 
    hovermode="x unified"
)

# Aplica o tema global do case
apply_dochmo_theme(fig_final, height=580)

# Sobrescreve especificamente a margem superior (t) para 75 para dar espaço à legenda e título
fig_final.update_layout(margin=dict(t=75))

write_case_html(fig_final, "../graphics/g03_previsao_modelo_final.html")